In [ ]:
!pip install -q scikit-learn openpyxl transformers torch

import pandas as pd
import re
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score, accuracy_score
from scipy.sparse import hstack
import os
import joblib
import gc
import time
from sklearn.model_selection import train_test_split

# =====================================================
# SPEED OPTIMIZATIONS
# =====================================================

# Force CPU usage if GPU is slow/unavailable
torch.backends.cudnn.benchmark = True
torch.set_num_threads(4)

# =====================================================
# Paths for BAD dataset
# =====================================================
datasets_a = {
    "Train": "/content/BAD_train-binary.xlsx",
    "Validation": "/content/BAD_val-binary.xlsx",
    "Test": "/content/BAD_test-binary.xlsx"
}
datasets_b = {
    "Train": "/content/BAD_train-multi.xlsx",
    "Validation": "/content/BAD_val-multi.xlsx",
    "Test": "/content/BAD_test-multi.xlsx"
}
datasets_c = {
    "Train": "/content/BAD_train-multi.xlsx",
    "Validation": "/content/BAD_val-multi.xlsx",
    "Test": "/content/BAD_test-multi.xlsx"
}

# =====================================================
# ULTRA-FAST Text Cleaning
# =====================================================
class FastTextCleaner:
    def __init__(self):
        # Pre-compile regex for speed
        self.clean_pattern = re.compile(r"http\S+|@\w+|#\w+|\d+")
        self.char_pattern = re.compile(r"[^\w\s\u0980-\u09FF]")
        self.space_pattern = re.compile(r"\s+")

    def clean_bangla_text(self, text):
        text = str(text)
        text = self.clean_pattern.sub("", text)
        text = self.char_pattern.sub("", text)
        return self.space_pattern.sub(" ", text).strip()

# Global cleaner instance
cleaner = FastTextCleaner()

# =====================================================
# ULTRA-FAST L3Cube Feature Extractor
# =====================================================
class FastL3CubeFeatureExtractor:
    def __init__(self, model_name="l3cube-pune/mahahate-bert"):
        print("Loading BERT model...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()

        # Use GPU only if available and faster
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)
        self.model.half()  # Use FP16 for speed

        print(f"Model loaded on {self.device}")

    def extract_features(self, texts, max_length=64, batch_size=32):  # Reduced max_length and increased batch_size
        """Extract BERT features with aggressive speed optimizations"""

        print(f"Extracting BERT features for {len(texts)} texts...")

        # Limit dataset size for speed
        if len(texts) > 8000:
            print(f"Sampling {8000} texts from {len(texts)} for speed...")
            indices = np.random.choice(len(texts), 8000, replace=False)
            texts = [texts[i] for i in sorted(indices)]

        features = []

        with torch.no_grad():
            for i in range(0, len(texts), batch_size):
                if i % (batch_size * 10) == 0:
                    print(f"Processing batch {i//batch_size + 1}/{(len(texts) + batch_size - 1)//batch_size}")

                batch_texts = texts[i:i+batch_size]

                # Truncate texts for speed
                batch_texts = [str(text)[:200] for text in batch_texts]  # Limit text length

                inputs = self.tokenizer(
                    batch_texts,
                    max_length=max_length,
                    padding='max_length',
                    truncation=True,
                    return_tensors='pt'
                )

                inputs = {k: v.to(self.device) for k, v in inputs.items()}

                outputs = self.model(**inputs)
                cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
                features.extend(cls_embeddings)

                # Clear GPU memory
                del inputs, outputs
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        result = np.array(features, dtype=np.float32)  # Use float32 for memory
        print(f"BERT features shape: {result.shape}")
        return result

# =====================================================
# ULTRA-FAST TF-IDF Vectorizers (REDUCED FEATURES)
# =====================================================
print("Initializing TF-IDF vectorizers...")
word_vectorizer = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 2),
    max_features=1500,  # Reduced from 3000
    max_df=0.95,       # Skip very common words
    min_df=3           # Skip very rare words
)
char_vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 4),  # Reduced from (3,5)
    max_features=1500,   # Reduced from 3000
    max_df=0.95,
    min_df=3
)
combined_vectorizer = FeatureUnion([('word', word_vectorizer), ('char', char_vectorizer)])

# =====================================================
# ULTRA-FAST Training Function
# =====================================================
def ultra_fast_run_subtask(dataset_paths, subtask_name):
    print(f"\n ULTRA-FAST {subtask_name} 🚀")
    start_time = time.time()

    # Load data
    print("Loading datasets...")
    train_df = pd.read_excel(dataset_paths["Train"])
    val_df = pd.read_excel(dataset_paths["Validation"])
    test_df = pd.read_excel(dataset_paths["Test"])
    train_df = pd.concat([train_df, val_df], ignore_index=True)

    # Drop NaN rows
    train_df = train_df.dropna(subset=['cleaned', 'Label'])
    test_df = test_df.dropna(subset=['cleaned', 'Label'])

    print(f"Data loaded: {len(train_df)} train, {len(test_df)} test")

    # SPEED HACK: Sample large datasets
    if len(train_df) > 12000:
        print(f"Sampling training data from {len(train_df)} to 10000 for speed...")
        train_df = train_df.sample(n=10000, random_state=42).reset_index(drop=True)

    # Fast text cleaning
    print("Fast text cleaning...")
    train_df['clean_text'] = [cleaner.clean_bangla_text(text) for text in train_df['cleaned']]
    test_df['clean_text'] = [cleaner.clean_bangla_text(text) for text in test_df['cleaned']]

    # Label encoding
    label_enc = LabelEncoder()
    y_train = label_enc.fit_transform(train_df['Label'])
    y_test = label_enc.transform(test_df['Label'])

    print(f"Classes: {list(label_enc.classes_)}")

    # ULTRA-FAST TF-IDF features
    print("Extracting TF-IDF features...")
    tfidf_start = time.time()
    X_train_tfidf = combined_vectorizer.fit_transform(train_df['clean_text'])
    X_test_tfidf = combined_vectorizer.transform(test_df['clean_text'])
    print(f"TF-IDF completed in {time.time() - tfidf_start:.2f}s")
    print(f"TF-IDF shape: {X_train_tfidf.shape}")

    # ULTRA-FAST BERT features
    print("Extracting BERT features...")
    bert_start = time.time()
    bert_extractor = FastL3CubeFeatureExtractor()
    X_train_bert = bert_extractor.extract_features(train_df['clean_text'].tolist())
    X_test_bert = bert_extractor.extract_features(test_df['clean_text'].tolist())

    # Handle sampling mismatch - FIXED: Use .shape[0] instead of len() for sparse matrices
    if len(X_train_bert) != X_train_tfidf.shape[0]:
        # Take corresponding TF-IDF features
        X_train_tfidf = X_train_tfidf[:len(X_train_bert)]
        y_train = y_train[:len(X_train_bert)]
        print(f"Adjusted training size to {len(X_train_bert)}")

    print(f"BERT completed in {time.time() - bert_start:.2f}s")

    # Clear memory
    del bert_extractor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Combine features
    print("Combining TF-IDF + BERT features...")
    X_train_combined = hstack([X_train_tfidf, X_train_bert])
    X_test_combined = hstack([X_test_tfidf, X_test_bert])
    print(f"Combined features shape: {X_train_combined.shape}")

    # Clear intermediate variables
    del X_train_tfidf, X_test_tfidf, X_train_bert, X_test_bert
    gc.collect()

    # ULTRA-FAST Grid Search (reduced parameters)
    print("Fast hyperparameter tuning...")
    param_grid = {
        'n_estimators': [100, 200],           # Reduced from [100, 300]
        'max_depth': [20, None],              # Reduced from [10, 30, None]
        'min_samples_split': [2, 5],          # Same
        'min_samples_leaf': [1],              # Reduced from [1, 2]
        'max_features': ['sqrt']              # Reduced from ['sqrt', 'log2']
    }

    grid_rf = GridSearchCV(
        estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
        param_grid=param_grid,
        scoring='f1_macro',
        cv=2,  # Reduced from 3
        n_jobs=1,  # Avoid nested parallelism
        verbose=1
    )

    grid_search_start = time.time()
    grid_rf.fit(X_train_combined, y_train)
    model = grid_rf.best_estimator_
    print(f"Grid search completed in {time.time() - grid_search_start:.2f}s")

    print(f"\nBest Parameters: {grid_rf.best_params_}")
    print(f"Best CV Score: {grid_rf.best_score_:.4f}")

    # Predictions
    print("Making predictions...")
    preds = model.predict(X_test_combined)

    # Results
    print("\nClassification Report:")
    print(classification_report(y_test, preds, target_names=[str(cls) for cls in label_enc.classes_]))

    macro_f1 = f1_score(y_test, preds, average='macro')
    acc = accuracy_score(y_test, preds)
    print(f"Accuracy: {acc:.4f}, Macro F1: {macro_f1:.4f}")

    # Save model (optional)
    try:
        save_path = '/content/drive/MyDrive/bad_models'
        os.makedirs(save_path, exist_ok=True)
        joblib.dump(model, f"{save_path}/fast_rf_{subtask_name}.joblib")
        joblib.dump(label_enc, f"{save_path}/fast_label_encoder_{subtask_name}.joblib")
        print(f" Model saved for {subtask_name}")
    except:
        print(f" Could not save model for {subtask_name}")

    total_time = time.time() - start_time
    print(f" {subtask_name} completed in {total_time/60:.2f} minutes!")

    return macro_f1

# =====================================================
# RUN ALL SUBTASKS WITH SPEED MONITORING
# =====================================================
print("ULTRA-FAST BERT+RF EXECUTION STARTED ")
print("Target: Complete all subtasks in under 30 minutes")

total_start_time = time.time()

try:
    results = {}
    results['subtask_a'] = ultra_fast_run_subtask(datasets_a, "subtask_a")
    results['subtask_b'] = ultra_fast_run_subtask(datasets_b, "subtask_b")
    results['subtask_c'] = ultra_fast_run_subtask(datasets_c, "subtask_c")

    total_time = time.time() - total_start_time

    print(f"\n FINAL RESULTS ")
    print(f"Total execution time: {total_time/60:.2f} minutes")
    print("-" * 50)
    for subtask, score in results.items():
        print(f"{subtask.upper()}: F1 = {score:.4f}")
    print("-" * 50)
    avg_f1 = np.mean(list(results.values()))
    print(f"Average F1-Score: {avg_f1:.4f}")
    print(f" SUCCESS: Completed in {total_time/60:.2f} minutes! ")

except Exception as e:
    print(f" Error occurred: {str(e)}")
    import traceback
    traceback.print_exc()

Initializing TF-IDF vectorizers...
ULTRA-FAST BERT+RF EXECUTION STARTED 
Target: Complete all subtasks in under 30 minutes

 ULTRA-FAST subtask_a 🚀
Loading datasets...
Data loaded: 12742 train, 1416 test
Sampling training data from 12742 to 10000 for speed...
Fast text cleaning...
Classes: [np.int64(0), np.int64(1)]
Extracting TF-IDF features...
TF-IDF completed in 2.79s
TF-IDF shape: (10000, 3000)
Extracting BERT features...
Loading BERT model...
 Error occurred: There was a specific connection error when trying to load l3cube-pune/mahahate-bert:
401 Client Error: Unauthorized for url: https://huggingface.co/l3cube-pune/mahahate-bert/resolve/main/config.json (Request ID: Root=1-68ba7784-0e6135d70182f5000abd270c;720c5c1b-d410-42f0-98b9-64dbe816d0b4)

Invalid credentials in Authorization header


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_http.py", line 409, in hf_raise_for_status
    response.raise_for_status()
  File "/usr/local/lib/python3.12/dist-packages/requests/models.py", line 1026, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 401 Client Error: Unauthorized for url: https://huggingface.co/l3cube-pune/mahahate-bert/resolve/main/config.json

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py", line 478, in cached_files
    hf_hub_download(
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py", line 114, in _inner_fn
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py", line 1010, in hf_hub_download
    retur